# 04 · At-Delivery Prediction

Documents the delivery-stage feature set, model-selection logic, cross-validation, tuning boundary and verified held-out CatBoost result.

In [ ]:
import pandas as pd
df=pd.read_csv('../data/master_orders_clean_v2.csv',low_memory=False)
train=df[df.split=='train'].copy(); test=df[df.split=='test'].copy()

In [ ]:
BASE_NUMERIC=['n_items','n_distinct_sellers','n_distinct_products','n_product_categories','total_price','total_freight','avg_item_price','avg_product_weight_g','max_product_weight_g','avg_product_volume_cm3','max_product_volume_cm3','total_payment_value','n_payment_installments','n_payment_methods']
PLACEMENT_FE=['purchase_hour','purchase_day_of_week','purchase_month','is_weekend','promised_delivery_days','freight_to_price_ratio','avg_freight_per_item','same_customer_seller_state']
DELIVERY_FE=['delivery_time_days','delivery_vs_estimate_days','approval_delay_days','carrier_timestamp_anomaly','carrier_pickup_delay_days','shipping_transit_days','delivered_late','late_days','early_days']
LOW_CARD_CAT=['customer_state','primary_product_category','primary_payment_type','primary_seller_state']
HIGH_CARD_CAT=['customer_city','primary_seller_city']
AT_DELIVERY=BASE_NUMERIC+PLACEMENT_FE+DELIVERY_FE+LOW_CARD_CAT+HIGH_CARD_CAT
print(len(AT_DELIVERY),'delivery-stage features')

## Model development

The original delivery track compared balanced Logistic Regression, a Decision Tree and CatBoost under 5-fold stratified cross-validation. CatBoost was selected for stronger CV performance and its handling of high-cardinality categorical fields. Feature selection did not improve CV F1, so the full feature set was retained.

In [ ]:
cv_folds=pd.read_csv('../outputs/catboost_cv_folds.csv')
cv_folds

Optuna was used for a limited search. The best tested configuration used 180 iterations, depth 6, learning rate 0.08 and L2 regularization 3. Threshold selection used out-of-fold training probabilities; the selected threshold was 0.64.

## Verified final result

The saved held-out CatBoost predictions achieve **precision 0.515, recall 0.457, F1 0.484 and ROC-AUC 0.768**. Confusion matrix: TN 15,656; FP 1,055; FN 1,333; TP 1,121.